# Graph-Transformer Training Pipeline

**Objective**: Train an Advanced Graph-Transformer (TransformerConv + Node Identity + RWSE) on preprocessed QED Feynman diagrams to predict squared amplitudes as token sequences.

This notebook replicates `src/train/train_graph_mlp.py` as an executable case study.

In [ ]:
import math, time, os, sys, re
from pathlib import Path

import sympy
from sympy.parsing.sympy_parser import parse_expr
import threading

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import to_dense_batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv
import torch_geometric.transforms as T
from tqdm.notebook import tqdm

PROJECT_ROOT = Path(os.getcwd()).parent if 'Notebooks' in os.getcwd() else Path(os.getcwd())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')

## 1 - Configuration

In [ ]:
D_MODEL       = 256
GNN_HIDDEN    = 256
GNN_LAYERS    = 3
N_HEADS       = 8
N_DEC_LAYERS  = 4
D_FF          = 1024
DROPOUT       = 0.1
MAX_LEN       = 512
NODE_FEAT_DIM = 64
EDGE_FEAT_DIM = 32
RWSE_DIM      = 16
MAX_NODES     = 200
BATCH         = 32
EPOCHS        = 10
LR            = 1.5e-3
PAD_IDX       = 1

## 2 - Load Preprocessed Data

In [ ]:
data_dir = PROJECT_ROOT / 'preprocessed' / 'qed'

train_data = torch.load(data_dir / 'train_graphs.pt')
val_data   = torch.load(data_dir / 'val_graphs.pt')
test_data  = torch.load(data_dir / 'test_graphs.pt')

train_ds, vocab = train_data['dataset'], train_data['vocab']
val_ds   = val_data['dataset']
test_ds  = test_data['dataset']

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}, Vocab: {len(vocab)}')

## 3 - Compute Random Walk Positional Encodings (RWSE)

In [ ]:
print('Computing RWSE...')
rwse = T.AddRandomWalkPE(walk_length=RWSE_DIM, attr_name='rwse')
for ds in [train_ds, val_ds, test_ds]:
    for i in range(len(ds)):
        ds[i] = rwse(ds[i])
print('Done.')

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH)
test_loader  = DataLoader(test_ds,  batch_size=BATCH)

## 4 - Model Architecture

In [ ]:
class GraphTransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.node_emb = nn.Linear(NODE_FEAT_DIM, GNN_HIDDEN)
        self.idx_emb  = nn.Embedding(MAX_NODES, GNN_HIDDEN)
        self.rwse_emb = nn.Linear(RWSE_DIM, GNN_HIDDEN)
        self.convs = nn.ModuleList([
            TransformerConv(GNN_HIDDEN, GNN_HIDDEN, heads=4, concat=False, edge_dim=EDGE_FEAT_DIM, dropout=DROPOUT)
            for _ in range(GNN_LAYERS)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(GNN_HIDDEN) for _ in range(GNN_LAYERS)])
        self.proj  = nn.Linear(GNN_HIDDEN, D_MODEL)

    def forward(self, batch_data):
        x, edge_index, batch, edge_attr = batch_data.x, batch_data.edge_index, batch_data.batch, batch_data.edge_attr
        h = self.node_emb(x)
        counts = torch.bincount(batch)
        local_idx = torch.cat([torch.arange(c.item(), device=x.device) for c in counts])
        h = h + self.idx_emb(local_idx)
        if hasattr(batch_data, 'rwse'):
            h = h + self.rwse_emb(batch_data.rwse)
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index, edge_attr)
            h = norm(h + h_new)
        h_dense, mask = to_dense_batch(h, batch)
        return self.proj(h_dense), mask

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(-torch.arange(0, d_model, 2).float() * math.log(10000.0) / d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class AdvancedGraphTransformer(nn.Module):
    def __init__(self, tgt_vocab_size):
        super().__init__()
        self.encoder  = GraphTransformerEncoder()
        self.tgt_emb  = nn.Embedding(tgt_vocab_size, D_MODEL, padding_idx=PAD_IDX)
        self.pos_enc  = PositionalEncoding(D_MODEL)
        dec_layer = nn.TransformerDecoderLayer(D_MODEL, N_HEADS, D_FF, DROPOUT, batch_first=True, norm_first=True)
        self.decoder  = nn.TransformerDecoder(dec_layer, N_DEC_LAYERS)
        self.out_proj = nn.Linear(D_MODEL, tgt_vocab_size)

    def forward(self, batch_data, tgt, tgt_mask=None, tgt_pad_mask=None):
        memory, mem_mask = self.encoder(batch_data)
        t = self.pos_enc(self.tgt_emb(tgt))
        out = self.decoder(t, memory, tgt_mask=tgt_mask,
                           tgt_key_padding_mask=tgt_pad_mask,
                           memory_key_padding_mask=torch.logical_not(mem_mask))
        return self.out_proj(out)

model = AdvancedGraphTransformer(len(vocab)).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

## 5 - Training and Evaluation Functions

In [ ]:
def causal_mask(sz, device):
    return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()

def check_symbolic_equivalence(p_toks, t_toks):
    def toks_to_str(l):
        parts = []
        for t in l:
            tok = str(t)
            if tok in ('<bos>', '<eos>', '<pad>', '<unk>'):
                continue
            tok = tok.replace('<S>', 's_var').replace('<s>', 's_var')
            tok = tok.replace('<T>', 't_var').replace('<t>', 't_var')
            tok = tok.replace('<U>', 'u_var').replace('<u>', 'u_var')
            tok = tok.replace('<GAMMA>', 'gamma_var').replace('<gamma>', 'gamma_var')
            tok = tok.replace('SQUARE', '**2')
            tok = tok.replace('INDEX_', 'idx_')
            tok = tok.replace('MOMENTUM_', 'p_')
            tok = tok.replace('<', '').replace('>', '')
            parts.append(tok)
        return ' '.join(parts)
    p, t = toks_to_str(p_toks), toks_to_str(t_toks)
    if not p.strip() or not t.strip(): return False
    p_norm = re.sub(r'\s+', '', p)
    t_norm = re.sub(r'\s+', '', t)
    if p_norm == t_norm: return True
    result = [False]
    def _check():
        try:
            local_dict = {}
            expr_p = parse_expr(p, local_dict=local_dict)
            expr_t = parse_expr(t, local_dict=local_dict)
            if sympy.expand(expr_p - expr_t) == 0:
                result[0] = True
                return
            result[0] = (sympy.simplify(expr_p - expr_t) == 0)
        except Exception:
            pass
    worker = threading.Thread(target=_check)
    worker.daemon = True
    worker.start()
    worker.join(5.0)
    return result[0]

@torch.no_grad()
def greedy_decode_batch(model, batch_data, bos_idx, eos_idx, pad_idx, max_len, device):
    model.eval()
    memory, mem_mask = model.encoder(batch_data)
    mem_key_pad = torch.logical_not(mem_mask)
    batch_size = memory.size(0)
    ys = torch.full((batch_size, 1), bos_idx, dtype=torch.long, device=device)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
    for _ in range(max_len - 1):
        tgt_mask = causal_mask(ys.size(1), device)
        t = model.pos_enc(model.tgt_emb(ys))
        out = model.decoder(t, memory, tgt_mask=tgt_mask,
                            memory_key_padding_mask=mem_key_pad)
        logits = model.out_proj(out[:, -1, :])
        next_tok = logits.argmax(-1)
        next_tok = next_tok.masked_fill(finished, pad_idx)
        ys = torch.cat([ys, next_tok.unsqueeze(1)], dim=1)
        finished = finished | (next_tok == eos_idx)
        if finished.all():
            break
    return ys[:, 1:]

print('Functions defined.')

In [ ]:
def run_train_epoch(model, loader, criterion, optimizer, scheduler=None):
    model.train()
    total_loss, c_tok, n_tok = 0.0, 0, 0
    for batch in tqdm(loader, leave=False):
        batch   = batch.to(DEVICE)
        tgt_in  = batch.y[:, :-1]
        tgt_out = batch.y[:, 1:]
        tgt_mask = causal_mask(tgt_in.size(1), DEVICE)
        tgt_pad  = torch.eq(tgt_in, PAD_IDX)
        logits = model(batch, tgt_in, tgt_mask=tgt_mask, tgt_pad_mask=tgt_pad)
        loss   = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
        preds = logits.argmax(-1)
        mask  = torch.ne(tgt_out, PAD_IDX)
        c_tok += torch.eq(preds[mask], tgt_out[mask]).sum().item()
        n_tok += mask.sum().item()
    return total_loss / len(loader), (c_tok / n_tok * 100 if n_tok else 0)

def run_eval_epoch(model, loader, criterion, bos_idx, eos_idx, compute_metrics=True):
    model.eval()
    total_loss, c_seq, c_sym, n_seq = 0.0, 0, 0, 0
    for batch in tqdm(loader, leave=False):
        batch   = batch.to(DEVICE)
        tgt_in  = batch.y[:, :-1]
        tgt_out = batch.y[:, 1:]
        with torch.no_grad():
            tgt_mask = causal_mask(tgt_in.size(1), DEVICE)
            tgt_pad  = torch.eq(tgt_in, PAD_IDX)
            logits   = model(batch, tgt_in, tgt_mask=tgt_mask, tgt_pad_mask=tgt_pad)
            loss     = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            total_loss += loss.item()
            if not compute_metrics:
                continue
            preds = greedy_decode_batch(model, batch, bos_idx, eos_idx, PAD_IDX, MAX_LEN, DEVICE)
            for b in range(preds.shape[0]):
                p_eos = (preds[b] == eos_idx).nonzero(as_tuple=True)[0]
                p_len = p_eos[0].item() if len(p_eos) > 0 else preds.shape[1]
                t_eos = (tgt_out[b] == eos_idx).nonzero(as_tuple=True)[0]
                t_len = t_eos[0].item() if len(t_eos) > 0 else tgt_out.shape[1]
                p_seq = preds[b, :p_len]
                t_seq = tgt_out[b, :t_len]
                if p_len == t_len and torch.equal(p_seq, t_seq):
                    c_seq += 1
                else:
                    ptoks = [vocab.idx_to_token.get(i.item(), '') for i in p_seq]
                    ttoks = [vocab.idx_to_token.get(i.item(), '') for i in t_seq]
                    if check_symbolic_equivalence(ptoks, ttoks):
                        c_sym += 1
                n_seq += 1
    return (
        total_loss / len(loader),
        c_seq / n_seq * 100 if n_seq else 0,
        (c_seq + c_sym) / n_seq * 100 if n_seq else 0,
    )

print('Training/eval functions defined.')

## 6 - Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, steps_per_epoch=len(train_loader), epochs=EPOCHS, pct_start=0.3)

save_dir = PROJECT_ROOT / 'checkpoints'
save_dir.mkdir(parents=True, exist_ok=True)
ckpt = save_dir / 'advanced_graph_mlp_qed_best.pt'

bos_idx = vocab.token_to_idx.get('<bos>', 2)
eos_idx = vocab.token_to_idx.get('<eos>', 3)

best_val = float('inf')
history = []

print(f'Training on {DEVICE} for {EPOCHS} epochs')
print('=' * 80)

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_l, tr_t = run_train_epoch(model, train_loader, criterion, optimizer, scheduler)
    comp_mets = (ep % 5 == 0) or (ep == EPOCHS)
    vl_l, vl_s, vl_sym = run_eval_epoch(model, val_loader, criterion, bos_idx, eos_idx, compute_metrics=comp_mets)
    elapsed = time.time() - t0

    if comp_mets:
        print(f'Ep {ep:3d}/{EPOCHS} | '
              f'Train loss={tr_l:.4f} tok={tr_t:.1f}% | '
              f'Val loss={vl_l:.4f} seq={vl_s:.1f}% sym={vl_sym:.1f}% | '
              f'{elapsed:.1f}s')
    else:
        print(f'Ep {ep:3d}/{EPOCHS} | '
              f'Train loss={tr_l:.4f} tok={tr_t:.1f}% | '
              f'Val loss={vl_l:.4f} (skipped decoding) | '
              f'{elapsed:.1f}s')

    history.append({'epoch': ep, 'train_loss': tr_l, 'val_loss': vl_l, 'tok_acc': tr_t, 'seq_acc': vl_s, 'sym_acc': vl_sym})

    if vl_l < best_val:
        best_val = vl_l
        torch.save(model.state_dict(), ckpt)

print('\nTraining complete. Best checkpoint saved.')

## 7 - Test Evaluation

In [ ]:
model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
te_l, te_s, te_sym = run_eval_epoch(model, test_loader, criterion, bos_idx, eos_idx)

print('=' * 60)
print('  QED TEST RESULTS - Advanced Graph-Transformer (MLP)')
print('=' * 60)
print(f'  Test loss     : {te_l:.4f}')
print(f'  Greedy seq_acc: {te_s:.2f}%')
print(f'  Greedy sym_acc: {te_sym:.2f}%')

## 8 - Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, train_losses, label='Train Loss', marker='o', markersize=3)
ax1.plot(epochs, val_losses, label='Val Loss', marker='s', markersize=3)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curves')
ax1.legend()
ax1.grid(True, alpha=0.3)

metric_epochs = [h['epoch'] for h in history if h['seq_acc'] > 0]
seq_accs = [h['seq_acc'] for h in history if h['seq_acc'] > 0]
sym_accs = [h['sym_acc'] for h in history if h['sym_acc'] > 0]

ax2.plot(metric_epochs, seq_accs, label='Seq Accuracy', marker='o')
ax2.plot(metric_epochs, sym_accs, label='Sym Accuracy', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()